# Parte 6 — Deployment
### Fine-tuning Completo · Feature Extraction · LoRA

Este notebook despliega cualquiera de los tres modelos como una API REST y la consume, **todo en el mismo kernel**.

```
Celda 1  →  instalar dependencias
Celda 2  →  configuración
Celda 3  →  cargar modelo
Celda 4  →  prueba rápida sin servidor
Celda 5  →  definir FastAPI app
Celda 6  →  levantar servidor en background
Celda 7  →  health check
Celda 8  →  predicción individual
Celda 9  →  predicción en batch
Celda 10 →  guardar client.py
```

## 1. Instalación

In [ ]:
!pip install fastapi uvicorn nest_asyncio requests -q

## 2. Configuración

Cambia `HF_REPO` para elegir el modelo a desplegar.

In [1]:
HF_REPO = "jdmartinev/imdb-distilbert-full"        # full fine-tuning
# HF_REPO = "jdmartinev/imdb-distilbert-head-only"  # feature extraction
# HF_REPO = "jdmartinev/imdb-distilbert-lora"       # LoRA

HOST       = "0.0.0.0"
PORT       = 8081
MAX_LENGTH = 256
ID2LABEL   = {0: "negative", 1: "positive"}
LABEL2ID   = {"negative": 0, "positive": 1}

## 3. Cargar el modelo

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
model     = AutoModelForSequenceClassification.from_pretrained(
    HF_REPO, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)
model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: {HF_REPO}")
print(f"Device: {device}  |  Parámetros: {n_params/1e6:.1f}M")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo: jdmartinev/imdb-distilbert-full
Device: cuda  |  Parámetros: 67.0M


## 4. Prueba rápida sin servidor

Verificamos que el modelo funciona antes de levantar la API.

In [4]:
def predict(text: str) -> list[dict]:
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH,
    ).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    values, indices = probs.topk(2)
    return [
        {"label": ID2LABEL[idx.item()], "score": round(val.item(), 4)}
        for val, idx in zip(values, indices)
    ]


samples = [
    "This movie was absolutely brilliant! The acting was superb.",
    "Terrible film. Boring plot, awful acting, complete waste of time.",
    "It had some interesting moments but ultimately failed to deliver.",
]

print(f"{'Reseña':<60}  {'Label':<12}  {'Score'}")
print("-" * 82)
for text in samples:
    top     = predict(text)[0]
    preview = (text[:57] + "...") if len(text) > 60 else text.ljust(60)
    icon    = "✅" if top["label"] == "positive" else "❌"
    print(f"{preview}  {icon} {top['label']:<10}  {top['score']:.4f}")

Reseña                                                        Label         Score
----------------------------------------------------------------------------------
This movie was absolutely brilliant! The acting was superb.   ✅ positive    0.9786
Terrible film. Boring plot, awful acting, complete waste ...  ❌ negative    0.9742
It had some interesting moments but ultimately failed to ...  ❌ negative    0.8355


## 5. Definir la API

In [5]:
import traceback
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(
    title="IMDB Sentiment Classifier",
    description="Análisis de sentimiento: negative / positive",
    version="1.0",
)


class TextInput(BaseModel):
    text: str


class BatchInput(BaseModel):
    texts: list[str]


@app.get("/health")
def health():
    return {"status": "ok", "model": HF_REPO, "device": str(device)}


@app.get("/model")
def model_info():
    return {
        "hub_repo":   HF_REPO,
        "labels":     list(ID2LABEL.values()),
        "parameters": f"{n_params/1e6:.1f}M",
        "device":     str(device),
    }


@app.post("/predict")
def predict_endpoint(body: TextInput):
    if not body.text.strip():
        raise HTTPException(status_code=400, detail="text cannot be empty")
    try:
        return JSONResponse(content={
            "text":        body.text,
            "predictions": predict(body.text),
        })
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch")
def predict_batch_endpoint(body: BatchInput):
    if not body.texts:
        raise HTTPException(status_code=400, detail="texts list cannot be empty")
    if len(body.texts) > 64:
        raise HTTPException(status_code=400, detail="maximum 64 texts per batch")
    try:
        inputs = tokenizer(
            body.texts, return_tensors="pt",
            truncation=True, max_length=MAX_LENGTH, padding=True,
        ).to(device)
        with torch.inference_mode():
            probs = torch.softmax(model(**inputs).logits, dim=-1)
        results = []
        for text, row in zip(body.texts, probs):
            values, indices = row.topk(2)
            results.append({
                "text": text,
                "predictions": [
                    {"label": ID2LABEL[idx.item()], "score": round(val.item(), 4)}
                    for val, idx in zip(values, indices)
                ],
            })
        return JSONResponse(content={"results": results})
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


print("FastAPI app definida.")

FastAPI app definida.


## 6. Levantar el servidor en background

`nest_asyncio` resuelve el conflicto entre el event loop de Jupyter y el de uvicorn.  
El servidor corre en un thread separado — el kernel queda libre para las celdas del cliente.

In [6]:
import uvicorn
import threading
import time
import nest_asyncio

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host=HOST, port=PORT, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)   # esperar a que uvicorn arranque

print(f"Servidor corriendo en http://{HOST}:{PORT}")
print(f"Docs interactivos:    http://{HOST}:{PORT}/docs")

Servidor corriendo en http://0.0.0.0:8081
Docs interactivos:    http://0.0.0.0:8081/docs


## 7. Health check

In [7]:
import requests

SERVER_URL = f"http://localhost:{PORT}"

resp = requests.get(f"{SERVER_URL}/health")
print(resp.json())

resp = requests.get(f"{SERVER_URL}/model")
print(resp.json())

{'status': 'ok', 'model': 'jdmartinev/imdb-distilbert-full', 'device': 'cuda'}
{'hub_repo': 'jdmartinev/imdb-distilbert-full', 'labels': ['negative', 'positive'], 'parameters': '67.0M', 'device': 'cuda'}


## 8. Predicción individual

In [ ]:
review = "This film was a complete masterpiece. I was on the edge of my seat the whole time."

resp = requests.post(f"{SERVER_URL}/predict", json={"text": review})
resp.raise_for_status()

result = resp.json()
print(f"Reseña: {result['text']}\n")
for p in result["predictions"]:
    bar  = "█" * int(p["score"] * 40)
    icon = "✅" if p["label"] == "positive" else "❌"
    print(f"  {icon} {p['label']:<12} {p['score']:.2%}  {bar}")

## 9. Predicción en batch

In [ ]:
reviews = [
    "This film was absolutely brilliant! The acting was superb.",
    "Terrible movie. Boring plot, awful acting, complete waste of time.",
    "It had some interesting moments but ultimately failed to deliver.",
    "One of the best films I've seen in years. A must-watch for everyone.",
    "I fell asleep halfway through. Nothing interesting happens at all.",
]

resp = requests.post(f"{SERVER_URL}/predict/batch", json={"texts": reviews})
resp.raise_for_status()

print(f"{'Reseña':<60}  {'Label':<12}  {'Score'}")
print("-" * 82)
for item in resp.json()["results"]:
    top     = item["predictions"][0]
    preview = (item["text"][:57] + "...") if len(item["text"]) > 60 else item["text"].ljust(60)
    icon    = "✅" if top["label"] == "positive" else "❌"
    print(f"{preview}  {icon} {top['label']:<10}  {top['score']:.4f}")

## 10. Guardar client.py

Script standalone para llamar al servidor desde cualquier máquina.

In [8]:
client_code = """\
#!/usr/bin/env python
\"\"\"Cliente para el servidor de análisis de sentimiento IMDB.

Uso:
    python client.py \"This movie was great!\"
    python client.py \"Terrible film.\" --url https://<id>-8081.cloudspaces.litng.ai
    python client.py --batch reviews.txt
\"\"\"
import argparse
import requests


def predict_one(url: str, text: str) -> None:
    resp = requests.post(f"{url}/predict", json={"text": text})
    resp.raise_for_status()
    result = resp.json()
    print(f"\\nReseña: {result['text']}\\n")
    for p in result["predictions"]:
        bar  = "\u2588" * int(p["score"] * 40)
        icon = "\u2705" if p["label"] == "positive" else "\u274c"
        print(f"  {icon} {p['label']:<12} {p['score']:.2%}  {bar}")


def predict_batch(url: str, filepath: str) -> None:
    with open(filepath) as f:
        texts = [line.strip() for line in f if line.strip()]
    resp = requests.post(f"{url}/predict/batch", json={"texts": texts})
    resp.raise_for_status()
    print(f"{'Reseña':<60}  {'Label':<12}  {'Score'}")
    print("-" * 82)
    for item in resp.json()["results"]:
        top     = item["predictions"][0]
        preview = (item["text"][:57] + "...") if len(item["text"]) > 60 else item["text"].ljust(60)
        icon    = "\u2705" if top["label"] == "positive" else "\u274c"
        print(f"{preview}  {icon} {top['label']:<10}  {top['score']:.4f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("text",    nargs="?",     help="Reseña a clasificar")
    parser.add_argument("--batch", metavar="FILE", help="Archivo con una reseña por línea")
    parser.add_argument("--url",   default="http://localhost:8081")
    args = parser.parse_args()

    if args.batch:
        predict_batch(args.url, args.batch)
    elif args.text:
        predict_one(args.url, args.text)
    else:
        parser.print_help()


if __name__ == \"__main__\":
    main()
"""

with open("client.py", "w") as f:
    f.write(client_code)

print("client.py guardado.")
print()
print("Uso desde terminal:")
print('  python client.py "This movie was great!"')
print('  python client.py --batch reviews.txt')
print('  python client.py "Great film!" --url https://<id>-8081.cloudspaces.litng.ai')

client.py guardado.

Uso desde terminal:
  python client.py "This movie was great!"
  python client.py --batch reviews.txt
  python client.py "Great film!" --url https://<id>-8081.cloudspaces.litng.ai
